In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.offline as pyo
import cvxpy as cp

from scipy.optimize import minimize

pyo.init_notebook_mode(connected=True)
pd.options.plotting.backend = 'plotly'

In [2]:
stocklist = ['^GSPC', 'GC=F', 'TLT', 'BTC-USD']
stocklist

['^GSPC', 'GC=F', 'TLT', 'BTC-USD']

In [3]:
prices = yf.download(stocklist, start='2015-01-01')['Close']
prices

[*********************100%***********************]  4 of 4 completed


Ticker,BTC-USD,GC=F,TLT,^GSPC
Date,,,,
2015-01-01,314.248993,NaN,NaN,NaN
2015-01-02,315.032013,1186.000000,94.399391,2058.199951
2015-01-03,281.082001,NaN,NaN,NaN
2015-01-04,264.195007,NaN,NaN,NaN
2015-01-05,274.473999,1203.900024,95.882248,2020.579956
...,...,...,...,...
2026-02-04,73019.703125,4920.399902,86.540001,6882.720215
2026-02-05,62702.097656,4861.399902,87.480003,6798.399902
2026-02-06,70555.390625,4979.799805,87.540001,6932.299805


In [4]:
log_returns = np.log(prices/prices.shift(1)).dropna()
log_returns

Ticker,BTC-USD,GC=F,TLT,^GSPC
Date,,,,
2015-01-06,0.041796,0.012711,0.017857,-0.008933
2015-01-07,0.028073,-0.007161,-0.001976,0.011563
2015-01-08,-0.038046,-0.001819,-0.013332,0.017730
2015-01-09,0.024607,0.006270,0.010894,-0.008439
2015-01-13,-0.170306,0.001297,0.000000,-0.002582
...,...,...,...,...
2026-01-30,-0.005133,-0.120657,-0.005608,-0.004311
2026-02-03,-0.039600,0.059054,0.002423,-0.008439
2026-02-04,-0.035171,0.003400,-0.002539,-0.005085


# Training Data

In [5]:
train = log_returns.iloc[:int(0.7 * len(log_returns))]
test = log_returns.iloc[int(0.7 * len(log_returns)):]

In [6]:
mean = train.mean()
cov = train.cov()

N_assets = len(mean)

bounds = [(0,1)] * N_assets
constraints = ({'type':'eq', 
                'fun': lambda w: w.sum()-1},)
init_w = np.repeat(1/N_assets, N_assets)

# Mean-Sharpe

In [7]:
def neg_sharpe(w):
    ret = w @ mean
    vol = np.sqrt(w @ cov @ w)
    return -ret / vol

res_sharpe = minimize(neg_sharpe, init_w, bounds=bounds, constraints=constraints)
w_sharpe = res_sharpe.x

In [8]:
w_sharpe

array([0.04216867, 0.15588935, 0.34628156, 0.45566043])

# Mean-Sortino

In [9]:
def neg_sortino(w):
    pr = train.values @ w
    downside = np.minimum(pr, 0)
    downside_dev = np.sqrt(np.mean(downside**2))
    return -np.mean(pr) / downside_dev

res_sortino = minimize(neg_sortino, init_w, bounds=bounds, constraints=constraints)
w_sortino = res_sortino.x

In [10]:
w_sortino

array([0.03883985, 0.1834415 , 0.31848993, 0.45922872])

# Mean-CVaR

In [15]:
alpha = 0.95
T = len(train)

w = cp.Variable(N_assets)
eta = cp.Variable()

pr = train.values @ w

In [16]:
cvar = eta + (1/((1-alpha)*T)) * cp.sum(cp.pos(-pr - eta))

In [72]:
target_ret = 0.0010
problem = cp.Problem(cp.Minimize(cvar),
                     [cp.sum(w) == 1, w >= 0,
                     cp.sum(train.values @ w)/T >= target_ret])

problem.solve()
w_cvar = w.value

In [73]:
w_cvar

array([ 9.88748556e-01,  2.51589570e-11, -1.12503639e-11,  1.12514442e-02])